# gramlab — full English Wikipedia index (CPU, ≈3–5 h)
No input needed. Streams every English Wikipedia article (20231101 dump), keeps the lead paragraph (≤100 words) of each article
with at least 60 words, builds the BM25 index over all of them (~4–5M passages, ~4 GB), and scores it on the same 3,000 queries
as before (expect lower recall than over 659k passages: more look-alikes). Output: `work/bm25_full` — the chat notebook prefers it
when attached.

In [ ]:
import os, subprocess, sys, time, glob, shutil, json
T0 = time.time()
def sh(cmd, must=True):
    print('\n$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if must and r.returncode != 0:
        raise SystemExit(f'command failed ({r.returncode}): {cmd}')
os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    sh('git clone --depth 1 https://github.com/furevych/gramlab.git')
os.chdir('/kaggle/working/gramlab')
os.makedirs('work', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('data', exist_ok=True)

sh('python scripts/fetch_wikipedia.py --lang en --stream --lead_only 100 --out data/enwiki_leads.jsonl')
sh('python -m gramlab.retrieval build --wiki data/enwiki_leads.jsonl --out work/bm25_full --max_words 100')
q = sorted(glob.glob('/kaggle/input/**/work/bm25/eval_queries.jsonl', recursive=True))
if q:
    shutil.copy(q[0], 'work/bm25_full/eval_queries.jsonl')
    sh('python -m gramlab.retrieval eval --wiki data/enwiki_leads.jsonl --index work/bm25_full --out results/retrieval_bm25_full.json', must=False)
for s in ['Tell me about Belgrade.', 'Which river flows through Vienna?', 'who is churchhill']:
    sh(f'python -m gramlab.retrieval search --index work/bm25_full --q "{s}" -k 3', must=False)
os.remove('data/enwiki_leads.jsonl')
shutil.rmtree('.git', ignore_errors=True)
os.system('du -sh work/bm25_full')
print(f'total {(time.time()-T0)/60:.0f} min')